# NeurIPS Diffusion Evaluation

Train diffusion model, compute FID (AE or GyroSwin latent space), evaluate warm restarts, test FID-vs-convergence.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os

sys.path.append("..")
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [ ]:
import omegaconf, yaml
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

from neugk.diffusion import get_diffusion_runner

from neurips_diff_eval import (
    compute_statistics,
    compute_fid,
    extract_gyroswin_latents,
    to_model_space,
    from_model_space,
)

## 0. Configuration

In [ ]:
DATA_PATH = "/local00/bioinf/galletti/preprocessed_kvikio"
AE_CHECKPOINT = "/restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327/best.pth"
# Same scaling-law GyroSwin checkpoint used by neurips_fid_gyroswin_latents.ipynb
GYROSWIN_CHECKPOINT = "/restricteddata/ukaea/checkpoints/scaling_law/gyroswin_xxl_fluxavg_cond_nodrop_l1"
GKW_RAW_DIR = "/restricteddata/ukaea/gyrokinetics/raw"

# pretrained diffusion model
PRETRAINED_DIR = "/restricteddata/ukaea/checkpoints/neurips26/DIFF_FLOW/20260412_180101_948/"

ID_VAL = [
    "iteration_262.h5",
    "iteration_135.h5",
    "iteration_8.h5",
    "iteration_232.h5",
    "iteration_148.h5",
    "iteration_115.h5",
]
OOD_VAL = [f"ood_iteration_{i}.h5" for i in range(5)]
TRAIN_TRAJS = "iteration_{0-5,7-12,14-31,33-82,84-99}.h5"

N_EPOCHS = 10
BATCH_SIZE = 256
LR = 2e-3
VAL_EVERY = 30
N_DENOISING_STEPS = 15

MINIBATCH_OT = True
NOISE_DISTRIBUTION = "gaussian"  # "gaussian" or "mixture"
CONTINUOUS_TIME = True

FID_N_COMPONENTS = 512
FID_FRAC = 0.5             # fraction of the val set used for FID feature collection
VAL_SUBSAMPLE = 10         # runner valset stride (1=full, 5≈20%); used by runner.evaluate / probes
GEN_BATCH_SIZE = 32

In [ ]:
# --- paper-figure helpers (style + save/load) -----------------------------
from neurips_paper_plots import (
    apply_paper_style, save_fig, save_method_results,
    pretty_scatter, COLOR_FLUX, COLOR_KY, COLOR_QY, COLOR_REF, COLOR_GYROFLOW, darken,
)

apply_paper_style()
RESULTS_DIR = "/system/user/galletti/git/neural-gyrokinetics-gitlab/notebooks/figs/paper"
METHOD_NAME = "GyroFlow"
print(f"paper figures will be saved to {RESULTS_DIR}")


## 1. Train Diffusion Model

In [ ]:
print(f"loading pretrained from {PRETRAINED_DIR}")
pretrained_cfg = omegaconf.OmegaConf.load(os.path.join(PRETRAINED_DIR, "config.yaml"))
pretrained_cfg.output_path = PRETRAINED_DIR
pretrained_cfg.dataset.path = DATA_PATH
pretrained_cfg.ae_checkpoint = os.path.dirname(AE_CHECKPOINT)
pretrained_cfg.dataset.gds_override = True
pretrained_cfg.dataset.validation_trajectories = ID_VAL + OOD_VAL  # include OOD so latent_dashboard's diffusion sampler can resolve OOD trajs
pretrained_cfg.dataset.val_subsample = VAL_SUBSAMPLE
pretrained_cfg.dataset.eval_cond_filters = {}
pretrained_cfg.validation.probe = {"targets": []}
pretrained_cfg.logging.writer = None
pretrained_cfg.logging.tqdm = True
pretrained_cfg.training.num_workers = 0
pretrained_cfg.training.pin_memory = False
pretrained_cfg.ddp.enable = False
pretrained_cfg.deepspeed.enable = False
runner = get_diffusion_runner(rank=0, cfg=pretrained_cfg, world_size=1)
torch.use_deterministic_algorithms(False)
ckp = torch.load(
    os.path.join(PRETRAINED_DIR, "best.pth"),
    map_location=runner.device,
    weights_only=False,
)
runner.model.load_state_dict(ckp["model_state_dict"])
runner.model.eval()
print(f"--> loaded diffusion weights from epoch {ckp.get('epoch', '?')}")
print(f"Model: {sum(p.numel() for p in runner.model.parameters())/1e6:.1f}M params")
print(f"Train: {len(runner.trainset)}, Val: {sum(len(v) for v in runner.valsets)}")

# --- in-memory patch: OOD metadata.pkl on disk lacks the `flux` key (owned
#     by another user, can't rewrite). Inject `flux` into the loaded valset
#     metadata for any OOD entry so cyclone_diff._load_data doesn't KeyError.
import numpy as _np
for _fi, _fpath in enumerate(runner.valsets[0].files):
    _meta = runner.valsets[0].metadata[_fi]
    if "flux" in _meta:
        continue
    _name = os.path.basename(str(_fpath)).replace("_ifft_realpotens", "")
    if _name.startswith("ood_iteration_"):
        _k = _name[len("ood_iteration_"):]
        _raw = os.path.join(GKW_RAW_DIR, "ood", f"iteration_{_k}")
    else:
        _raw = os.path.join(GKW_RAW_DIR, _name)
    try:
        _fl = _np.loadtxt(os.path.join(_raw, "fluxes.dat"))[:, 1]
        _t  = _np.loadtxt(os.path.join(_raw, "time.dat"))
        _ts = _np.asarray(_meta["timesteps"])
        _ix = [_np.isclose(_t, t).nonzero()[0][0] for t in _ts]
        _meta["flux"] = _np.clip(_fl[_ix], a_min=0.0, a_max=None)
        if "flux_mean" not in _meta: _meta["flux_mean"] = float(_meta["flux"].mean())
        if "flux_std"  not in _meta: _meta["flux_std"]  = float(_meta["flux"].std())
        if "flux_min"  not in _meta: _meta["flux_min"]  = float(_meta["flux"].min())
        if "flux_max"  not in _meta: _meta["flux_max"]  = float(_meta["flux"].max())
        if "flux_var"  not in _meta: _meta["flux_var"]  = float(_meta["flux"].var())
        print(f"  injected `flux` for {_name} (n={len(_meta['flux'])})")
    except Exception as _e:
        print(f"  could not inject `flux` for {_name}: {_e}")



In [ ]:
raw = runner.trainset.__getitem__(100, get_normalized=False, override_latens=True)
print(f"raw std: {raw.df.std():.4f}, raw mean: {raw.df.mean():.4f}")
scale, shift = runner.trainset._get_scale_shift(0, "df", raw.df)
print(f"shift shape: {shift.shape}, scale shape: {scale.shape}")
print(f"shift: {shift.squeeze()}")
print(f"scale: {scale.squeeze()}")
manual = (raw.df - shift) / scale
print(f"manual normalized std: {manual.std():.4f}")

In [ ]:
from neugk.plot_utils import plot_nd

sample = runner.trainset.__getitem__(100, get_normalized=True, override_latens=True)
df_in = sample.df.unsqueeze(0).to(runner.device)
cond = sample.conditioning.unsqueeze(0).to(runner.device)

ae = runner.autoencoder
ae.eval()
with torch.no_grad():
    recon = ae(df_in, condition=cond)["df"]

df_gt = df_in[0].cpu()
df_rec = recon[0].cpu()

print(f"Input shape: {df_gt.shape}, Recon shape: {df_rec.shape}")
print(f"AE recon MSE: {(df_gt - df_rec).pow(2).mean():.6f}")
print(f"AE recon rel err: {(df_gt - df_rec).norm() / df_gt.norm():.4f}")

_ = plot_nd(df_gt, df_rec, to_wandb=False)

In [ ]:
sample = runner.trainset.__getitem__(0, get_normalized=True, override_latens=True)
cond = sample.conditioning.unsqueeze(0).to(runner.device)

runner.model.eval()
with torch.no_grad():
    gen_out = runner.sample(cond, latent_only=False, steps=N_DENOISING_STEPS)

df_gt = sample.df.cpu()
df_gen = gen_out["df"][0].cpu()

print(f"GT shape: {df_gt.shape}, Gen shape: {df_gen.shape}")

_ = plot_nd(df_gt, df_gen, to_wandb=False)

## 1.5 Flux UQ (ID)

In [ ]:
runner.model.eval()
log_metrics, val_plots, _ = runner.evaluate(epoch=0, evaluate_probing=False, no_save=True)

In [ ]:
val_plots.keys()

In [ ]:
for k, v in sorted(log_metrics.items()):
    print(f"  {k}: {v:.4f}")

display(val_plots["avg_flux_UQ"].image)

In [ ]:
from neurips_diff_eval import (
    collect_latents,
    generate_latents,
    fit_probes,
    plot_probes,
    encode_valset,
    plot_val_probe,
)

PROBE_N_COMPONENTS = 256
PROBE_ALPHA = 1.0
PROBE_SUBSAMPLE_FRAC = 0.10   # 10% of the training latents

_cond_keys = sorted(runner.cfg.model.conditioning)
X_ae, y_flux, C_train = collect_latents(runner.trainset.precomputed_latents, _cond_keys)
print(f"training set: {X_ae.shape[0]} samples, latent dim={X_ae.shape[1]}")

PROBE_SUBSAMPLE = max(1, int(PROBE_SUBSAMPLE_FRAC * len(X_ae)))
if PROBE_SUBSAMPLE and PROBE_SUBSAMPLE < len(X_ae):
    idx = np.random.choice(len(X_ae), PROBE_SUBSAMPLE, replace=False)
    X_ae, y_flux, C_train = X_ae[idx], y_flux[idx], C_train[idx]
    print(f"subsampled to {PROBE_SUBSAMPLE} samples")

X_gen = generate_latents(runner, C_train, batch_size=GEN_BATCH_SIZE, steps=N_DENOISING_STEPS)

probes = fit_probes(
    X_ae,
    X_gen,
    y_flux,
    _cond_keys,
    C_train,
    n_components=PROBE_N_COMPONENTS,
    alpha=PROBE_ALPHA,
)
print(
    f"PCA: {X_ae.shape[1]} -> {probes['pca'].n_components_} ({probes['pca'].explained_variance_ratio_.sum():.1%} var)"
)
print(f"flux probe — RMSE ae: {probes['flux']['rmse_ae']:.4f}, RMSE gen: {probes['flux']['rmse_gen']:.4f}")
for k in _cond_keys:
    print(f"  {k} — RMSE ae: {probes['cond']['rmse'][k]['ae']:.4f}, RMSE gen: {probes['cond']['rmse'][k]['gen']:.4f}")

_ = plot_probes(y_flux, probes, _cond_keys, C_train, X_ae, X_gen)

X_val_ae, C_val, y_val_gt, val_fi = encode_valset(
    runner.valsets[0],
    runner.autoencoder,
    _cond_keys,
    runner.device,
    batch_size=GEN_BATCH_SIZE,
)
X_val_gen = generate_latents(runner, C_val, batch_size=GEN_BATCH_SIZE, steps=N_DENOISING_STEPS)

pca = probes["pca"]
pred_val_ae = probes["flux"]["probe_ae"].predict(pca.transform(X_val_ae))
pred_val_gen = probes["flux"]["probe_gen"].predict(pca.transform(X_val_gen))
rmse_val_ae = np.sqrt(((y_val_gt - pred_val_ae) ** 2).mean())
rmse_val_gen = np.sqrt(((y_val_gt - pred_val_gen) ** 2).mean())
print(f"flux probe (val) — RMSE ae: {rmse_val_ae:.4f}, RMSE gen: {rmse_val_gen:.4f}")

_ = plot_val_probe(
    runner.valsets[0],
    pred_val_ae,
    pred_val_gen,
    y_val_gt,
    val_fi,
    rmse_val_ae,
    rmse_val_gen,
)

In [ ]:
# === paper figure: flux probe (generative latents only) ====================
# Uses the val-set predictions already computed above:
#   y_val_gt        — ground-truth scalar flux per val sample
#   pred_val_gen    — diffusion-latent → flux probe prediction
# Outputs:  <RESULTS_DIR>/probe_flux_gen.{pdf,png}
import numpy as np

apply_paper_style()
fig, ax = plt.subplots(figsize=(3.4, 3.4))
yt = np.asarray(y_val_gt, dtype=float)
yp = np.asarray(pred_val_gen, dtype=float)
mask = np.isfinite(yt) & np.isfinite(yp)
yt, yp = yt[mask], yp[mask]
rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
r = float(np.corrcoef(yt, yp)[0, 1]) if len(yt) > 2 else float("nan")
pretty_scatter(
    ax, yt, yp, color=COLOR_GYROFLOW, label="diffusion latents",
    diag=True, annot=f"RMSE = {rmse:.3f}\nr = {r:.3f}",
)
ax.set_xlabel("ground-truth Q")
ax.set_ylabel("predicted Q (probe on diff latents)")
ax.set_title("Flux probe — generative latents")
fig.tight_layout()
paths = save_fig(fig, "probe_flux_gen", RESULTS_DIR)
print(f"saved: {paths}")
plt.show()

PAPER_PROBE = {
    "y_true": yt, "y_pred_gen": yp, "rmse": rmse, "pearson": r,
}


## 2. FID Evaluation

- **`ae`**: FID in AE bottleneck (fast, no decode)
- **`gyroswin`**: FID using frozen GyroSwin encoder (Inception-FID analog)

In [ ]:
import pickle as _pickle
from neurips_gyroswin_eval import load_gyroswin_model
from neugk.dataset.cyclone_diff import CycloneAEDataset
from neugk.dataset.backend import KvikIOBackend

gyroswin_model, gs_cfg, _ = load_gyroswin_model(
    GYROSWIN_CHECKPOINT, dataset=runner.trainset, device=runner.device,
)
feature_fn = lambda b, device, **kw: extract_gyroswin_latents(gyroswin_model, b, device, **kw)
fid_label = "GyroSwin-FID"

_gs_stats_path = os.path.join(GYROSWIN_CHECKPOINT, "normalization_stats.pkl")
with open(_gs_stats_path, "rb") as f:
    gs_stats = _pickle.load(f)
print(f"loaded GyroSwin normalization stats from {_gs_stats_path}")

# Per-field zscore dict (CycloneAEDataset expects this shape, not a bare
gs_norm = {
    "df":   {"type": "zscore", "agg_axes": [1, 2, 3, 4, 5]},
    "flux": {"type": "zscore", "agg_axes": None},
    "phi":  {"type": "zscore", "agg_axes": None},
}
# Per-trajectory scalars only (timestep is per-snapshot, not a dataset cond key).
gs_cond_keys = [k for k in sorted(gs_cfg.model.conditioning) if k != "timestep"]

gs_valset = CycloneAEDataset(
    split="val", trajectories=ID_VAL,
    backend=KvikIOBackend(0, use_kvikio=False),
    active_keys=list(gs_cfg.dataset.active_keys),
    fields_to_load=["df", "phi", "flux"], probe_targets=[],
    path=DATA_PATH, random_seed=int(gs_cfg.seed),
    normalization=gs_norm,
    normalization_scope="dataset",
    spatial_ifft=bool(gs_cfg.dataset.spatial_ifft),
    bundle_seq_length=1,
    offset=int(gs_cfg.dataset.offset),
    separate_zf=bool(gs_cfg.dataset.separate_zf),
    num_workers=4,
    real_potens=bool(gs_cfg.dataset.real_potens),
    decouple_mu=bool(gs_cfg.dataset.get("norm_decouple_mu", False)),
    rank=0,
    conditions=gs_cond_keys,
    normalization_stats=gs_stats,
)
print(f"GyroSwin valset: {len(gs_valset)} samples (gs-bundled normalization)")

In [ ]:
from neugk.plot_utils import plot_nd

flat_idx_map = {(f, t): idx for idx, (f, t) in gs_valset.flat_index_to_file_and_tstep.items()}
fi = 0
meta = gs_valset.metadata[fi]
t_idx = 30  # input timestep; GyroSwin predicts df at t_idx + 1.
sample_in   = gs_valset[flat_idx_map[(fi, t_idx)]]
sample_next = gs_valset[flat_idx_map[(fi, t_idx + 1)]]
df_in = sample_in.df.unsqueeze(0).to(runner.device)

_cond_meta_map = {"itg": "ion_temp_grad", "dg": "density_grad"}
_gs_cond_keys = sorted(list(gs_cfg.model.conditioning))
_nontime = [k for k in _gs_cond_keys if k != "timestep"]
cond_kwargs = {
    k: torch.tensor(
        [float(np.squeeze(meta[_cond_meta_map.get(k, k)]))],
        dtype=torch.float32, device=runner.device,
    )
    for k in _nontime
}
if "timestep" in _gs_cond_keys:
    ts_val = float(meta["timesteps"][t_idx + gs_valset.offsets[fi]])
    cond_kwargs["timestep"] = torch.tensor([ts_val], dtype=torch.float32, device=runner.device)

fh = getattr(gyroswin_model, "flux_head", None)
if fh is not None and hasattr(fh, "condition_keys") and "timestep" not in fh.condition_keys:
    fh.condition_keys = sorted(list(fh.condition_keys) + ["timestep"])

gyroswin_model.eval()
with torch.no_grad():
    out = gyroswin_model(df_in, **cond_kwargs)
pred_df = out["df"][0].cpu()
gt_df_in   = sample_in.df.cpu()    # input @ t
gt_df_next = sample_next.df.cpu()  # target @ t+1 (autoregressive)

rel_l2_next = float((gt_df_next - pred_df).norm() / (gt_df_next.norm() + 1e-12))
rel_l2_same = float((gt_df_in   - pred_df).norm() / (gt_df_in.norm()   + 1e-12))
print(f"input shape: {gt_df_in.shape}, output shape: {pred_df.shape}")
print(f"recon rel err vs t+1 (correct AR target): {rel_l2_next:.4f}")
print(f"recon rel err vs t   (input, sanity):     {rel_l2_same:.4f}")
if rel_l2_next > 0.5:
    print("  !! WARNING rel L2 vs t+1 > 0.5 — weights may not have loaded or "
            "input normalization doesn't match what GyroSwin expects.")
fig = plot_nd(gt_df_next, pred_df, to_wandb=False)
fig.suptitle(f"GyroSwin: target df(t+1) (left) vs predicted (right) — rel L2 = {rel_l2_next:.3f}",
                fontsize=11, y=1.01)


In [ ]:
valset = gs_valset
t_start, t_step, n_snap = 80, 5, 32

_cond_meta_map_fid = {"itg": "ion_temp_grad", "dg": "density_grad"}
_gs_nontime = [k for k in gs_cond_keys if k != "timestep"]
flat_idx_map = {(f, t): idx for idx, (f, t) in valset.flat_index_to_file_and_tstep.items()}

import re as _re

all_dfs, all_conds, all_traj_idx, all_snap_labels = [], [], [], []
traj_labels = []

for fi in range(len(valset.files)):
    meta = valset.metadata[fi]
    fpath = valset.files[fi]
    m = _re.search(r"iteration_(\d+)", fpath)
    label = f"iter_{m.group(1)}" if m else f"f{fi}"

    cond_vals = [float(np.squeeze(meta[_cond_meta_map_fid.get(k, k)])) for k in _gs_nontime]

    snap_indices = [t_start + j * t_step for j in range(n_snap)]
    n_ts = len(meta["timesteps"])
    snap_indices = [s for s in snap_indices if s < n_ts]

    count = 0
    for si in snap_indices:
        t_idx = si - valset.offsets[fi]
        if t_idx < 0 or (fi, t_idx) not in flat_idx_map:
            continue
        sample = valset[flat_idx_map[(fi, t_idx)]]
        if sample.df is None:
            continue
        all_dfs.append(sample.df)
        all_conds.append(torch.tensor(cond_vals, dtype=torch.float32))
        all_traj_idx.append(len(traj_labels))
        all_snap_labels.append(f"{label}:{si}")
        count += 1

    if count >= 2:
        traj_labels.append(label)
        print(f"  {label}: {count} snapshots")
    else:
        for _ in range(count):
            all_dfs.pop(); all_conds.pop(); all_traj_idx.pop(); all_snap_labels.pop()

all_traj_idx = np.array(all_traj_idx)

# Extract features for both sources, identical sample batches.
_PER_TRAJ_SOURCES = (
    ("skip_deep",  {"source": "skip",       "decoder_level": -1, "pool": "amax", "cond_keys": _gs_nontime}),
    ("bottleneck", {"source": "bottleneck", "pool": "amax"}),
    ("flux_head",  {"source": "flux_head",  "flux_head_level": 1, "cond_keys": _gs_nontime}),
    ("phi",        {"source": "phi",        "pool": "amax", "cond_keys": _gs_nontime}),
)
features_per_source = {}
for src_name, src_kwargs in _PER_TRAJ_SOURCES:
    feats = []
    for i in tqdm(range(0, len(all_dfs), GEN_BATCH_SIZE),
                   desc=f"extract {src_name}"):
        batch_df = torch.stack(all_dfs[i:i+GEN_BATCH_SIZE])
        batch_cond = torch.stack(all_conds[i:i+GEN_BATCH_SIZE])
        feats.append(extract_gyroswin_latents(
            gyroswin_model, batch_df, device=runner.device,
            condition=batch_cond, **src_kwargs,
        ))
    features_per_source[src_name] = np.concatenate(feats)

n_traj = len(traj_labels)

# Build (FID matrix, cosine matrix) per source.
matrices_per_source = {}
for src_name, all_feats in features_per_source.items():
    n_comp = min(FID_N_COMPONENTS, all_feats.shape[0], all_feats.shape[1])
    pca_fid = PCA(n_components=n_comp).fit(all_feats)
    traj_feats = {ti: all_feats[all_traj_idx == ti] for ti in range(n_traj)}
    traj_feats_pca = {ti: pca_fid.transform(traj_feats[ti]) for ti in range(n_traj)}

    fid_matrix = np.full((n_traj, n_traj), np.nan)
    traj_stats = {ti: compute_statistics(traj_feats_pca[ti]) for ti in range(n_traj)}
    for i in range(n_traj):
        for j in range(i, n_traj):
            v = compute_fid(*traj_stats[i], *traj_stats[j])
            fid_matrix[i, j] = v; fid_matrix[j, i] = v

    feats_norm = all_feats / (np.linalg.norm(all_feats, axis=1, keepdims=True) + 1e-8)
    cos_matrix = feats_norm @ feats_norm.T

    matrices_per_source[src_name] = {
        "fid": fid_matrix, "cos": cos_matrix,
        "raw_dim": all_feats.shape[1], "pca_dim": n_comp,
        "explained_var": float(pca_fid.explained_variance_ratio_.sum()),
    }
    print(f"  {src_name}: PCA {all_feats.shape[1]} -> {n_comp} "
          f"({pca_fid.explained_variance_ratio_.sum():.1%} var)")

# 2x2 plot: row 0 = middle (bottleneck), row 1 = flux_head; col 0 = FID, col 1 = cosine.
traj_boundaries = []
cur = 0
for ti in range(n_traj):
    n = int((all_traj_idx == ti).sum())
    traj_boundaries.append((cur, cur + n, traj_labels[ti]))
    cur += n

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
_row_labels = (("bottleneck", "lowest middle (df_unet bottleneck)"),
               ("flux_head",  "flux_head (physics-targeted pre-MLP)"))

for ri, (src_name, src_title) in enumerate(_row_labels):
    M = matrices_per_source[src_name]
    fid_matrix, cos_matrix = M["fid"], M["cos"]

    vmax = np.nanmax(fid_matrix[~np.eye(n_traj, dtype=bool)]) if n_traj > 1 else 1.0
    im0 = axes[ri, 0].imshow(fid_matrix, cmap="YlOrRd", vmin=0, vmax=vmax)
    axes[ri, 0].set_xticks(range(n_traj))
    axes[ri, 0].set_xticklabels(traj_labels, rotation=45, ha="right", fontsize=9)
    axes[ri, 0].set_yticks(range(n_traj))
    axes[ri, 0].set_yticklabels(traj_labels, fontsize=9)
    for i in range(n_traj):
        for j in range(n_traj):
            if np.isfinite(fid_matrix[i, j]):
                axes[ri, 0].text(j, i, f"{fid_matrix[i,j]:.1f}", ha="center", va="center",
                                 fontsize=9, fontweight="bold",
                                 color="white" if fid_matrix[i, j] > 0.6 * vmax else "black")
    plt.colorbar(im0, ax=axes[ri, 0], label="FID", fraction=0.046)
    axes[ri, 0].set_title(f"{src_title}: pairwise FID per trajectory  "
                           f"(raw {M['raw_dim']} -> pca {M['pca_dim']}, "
                           f"{M['explained_var']:.0%} var)", fontsize=10)

    im1 = axes[ri, 1].imshow(cos_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
    for si, (s1, e1, l1) in enumerate(traj_boundaries):
        axes[ri, 1].axhline(s1 - 0.5, color="k", lw=0.5, alpha=0.5)
        axes[ri, 1].axvline(s1 - 0.5, color="k", lw=0.5, alpha=0.5)
        mid1 = (s1 + e1) / 2
        axes[ri, 1].text(-1.5, mid1, l1, ha="right", va="center", fontsize=7, fontweight="bold")
        axes[ri, 1].text(mid1, -1.5, l1, ha="center", va="bottom", fontsize=7, fontweight="bold", rotation=45)
        for sj, (s2, e2, l2) in enumerate(traj_boundaries):
            block_mean = cos_matrix[s1:e1, s2:e2].mean()
            axes[ri, 1].text((s2 + e2) / 2, mid1, f"{block_mean:.2f}",
                             ha="center", va="center", fontsize=7, fontweight="bold",
                             color="black",
                             bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))
    axes[ri, 1].axhline(cur - 0.5, color="k", lw=0.5, alpha=0.5)
    axes[ri, 1].axvline(cur - 0.5, color="k", lw=0.5, alpha=0.5)
    axes[ri, 1].set_xticks([]); axes[ri, 1].set_yticks([])
    plt.colorbar(im1, ax=axes[ri, 1], label="cosine sim", fraction=0.046)
    axes[ri, 1].set_title(f"{src_title}: pairwise cosine similarity per snapshot",
                           fontsize=10)

fig.tight_layout()

In [ ]:
runner.model.eval()

# Real samples: gs-normalized via gs_valset.
n = max(1, int(FID_FRAC * len(gs_valset)))
real_s = [gs_valset[i].df for i in tqdm(range(n), desc="Loading real (gs-norm)")]

_cond_meta_map = {"itg": "ion_temp_grad", "dg": "density_grad"}
_gs_nontime = [k for k in gs_cond_keys if k != "timestep"]
real_conds_list, real_fis = [], []
for idx in range(n):
    fi, _ = gs_valset.flat_index_to_file_and_tstep[idx]
    meta = gs_valset.metadata[fi]
    real_conds_list.append(torch.tensor(
        [float(np.squeeze(meta[_cond_meta_map.get(k, k)])) for k in _gs_nontime],
        dtype=torch.float32,
    ))
    real_fis.append(fi)

_diff_cond_keys = sorted(runner.cfg.model.conditioning)
diff_conds = []
for idx in range(n):
    fi, _ = gs_valset.flat_index_to_file_and_tstep[idx]
    meta = gs_valset.metadata[fi]
    diff_conds.append(torch.tensor(
        [float(np.squeeze(meta[_cond_meta_map.get(k, k)])) for k in _diff_cond_keys],
        dtype=torch.float32,
    ))
diff_conds = torch.stack(diff_conds)

# Gen samples: diffusion → denormalize via diff valset → re-normalize with gs_stats.
_gs_df_full = gs_stats["df"]["full"]
_gs_mean = torch.as_tensor(np.asarray(_gs_df_full["mean"]), dtype=torch.float32)
_gs_std  = torch.as_tensor(np.asarray(_gs_df_full["std"]),  dtype=torch.float32)
gen_s, gen_conds_list = [], []
for i in tqdm(range(0, n, GEN_BATCH_SIZE), desc="Generating (diff -> gs-norm)"):
    c = diff_conds[i : i + GEN_BATCH_SIZE].to(runner.device)
    with torch.no_grad():
        p = runner.sample(c, steps=N_DENOISING_STEPS, latent_only=False)
    for j in range(p["df"].shape[0]):
        df_diff_norm = p["df"][j].cpu()
        fi = real_fis[i + j]
        df_raw = runner.valsets[0].denormalize(fi, df=df_diff_norm)
        mean_b = _gs_mean.view(_gs_mean.shape + (1,) * (df_raw.ndim - _gs_mean.ndim))
        std_b  = _gs_std.view(_gs_std.shape  + (1,) * (df_raw.ndim - _gs_std.ndim))
        gen_s.append(((df_raw - mean_b) / std_b).to(df_diff_norm.dtype))
        gen_conds_list.append(real_conds_list[i + j])

print("Extracting GyroSwin features per latent source ...")
gs_features = {}
_SOURCES = (
    ("skip_deep",  {"source": "skip",       "decoder_level": -1, "pool": "amax", "cond_keys": _gs_nontime}),
    ("bottleneck", {"source": "bottleneck", "pool": "amax"}),
    ("flux_head",  {"source": "flux_head",  "flux_head_level": 1, "cond_keys": _gs_nontime}),
    ("phi",        {"source": "phi",        "pool": "amax", "cond_keys": _gs_nontime}),
)
for src_name, src_kwargs in _SOURCES:
    real_feats, gen_feats = [], []
    for i in tqdm(range(0, n, GEN_BATCH_SIZE), desc=f"feats:{src_name}"):
        r_batch = torch.stack(real_s[i:i+GEN_BATCH_SIZE])
        g_batch = torch.stack(gen_s[i:i+GEN_BATCH_SIZE])
        r_cond  = torch.stack(real_conds_list[i:i+GEN_BATCH_SIZE])
        g_cond  = torch.stack(gen_conds_list[i:i+GEN_BATCH_SIZE])
        real_feats.append(extract_gyroswin_latents(
            gyroswin_model, r_batch, device=runner.device, condition=r_cond, **src_kwargs,
        ))
        gen_feats.append(extract_gyroswin_latents(
            gyroswin_model, g_batch, device=runner.device, condition=g_cond, **src_kwargs,
        ))
    Xr = np.concatenate(real_feats, axis=0)
    Xg = np.concatenate(gen_feats,  axis=0)
    gs_features[src_name] = (Xr, Xg)
    print(f"  {src_name:11s}  raw real {Xr.shape}  raw gen {Xg.shape}")

# Default X_real/X_gen used by downstream cells = bottleneck (raw, pre-PCA).
X_real, X_gen = gs_features["bottleneck"]

print(f"Real: {X_real.shape}, Gen: {X_gen.shape}")

In [ ]:
def _safe_pca_fid(Xr, Xg, n_components):
    """PCA + FID with n_components clamped to min(n_samples, n_features)."""
    n_max = min(Xr.shape[0], Xr.shape[1])
    n_comp = min(n_components, n_max) if n_components else n_max
    if n_comp < Xr.shape[1]:
        _pca = PCA(n_components=n_comp)
        Xr_p = _pca.fit_transform(Xr)
        Xg_p = _pca.transform(Xg)
        ev = float(_pca.explained_variance_ratio_.sum())
    else:
        Xr_p, Xg_p, ev = Xr, Xg, 1.0
    m_r, s_r = compute_statistics(Xr_p)
    m_g, s_g = compute_statistics(Xg_p)
    return compute_fid(m_r, s_r, m_g, s_g), Xr_p.shape[1], ev

# Bottleneck FID (default downstream X_real / X_gen).
fid_global, _pca_dim, _ev = _safe_pca_fid(X_real, X_gen, n_components=128)
print(f"Global {fid_label} (bottleneck): {fid_global:.4f}  "
      f"(raw {X_real.shape[1]} -> pca {_pca_dim}, {_ev:.0%} var)")

# Per-source FID for the GyroSwin path: bottleneck (lowest middle) vs flux_head.
gs_fid_per_source = {}
if gs_features:
    print("\n  per-source FID (PCA-reduced):")
    for src_name, (Xr, Xg) in gs_features.items():
        fid_v, pca_dim, ev = _safe_pca_fid(Xr, Xg, n_components=128)
        gs_fid_per_source[src_name] = fid_v
        print(f"    {src_name:20s}  FID = {fid_v:>14.4f}   "
              f"(raw {Xr.shape[1]} -> pca {pca_dim}, {ev:.0%} var)")


In [ ]:
per_traj_fids = {}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

mr, mg = X_real.mean(0), X_gen.mean(0)
axes[0, 0].scatter(mr, mg, alpha=0.3, s=10, color="#264653")
lim = [min(mr.min(), mg.min()), max(mr.max(), mg.max())]
axes[0, 0].plot(lim, lim, "r--", alpha=0.8)
axes[0, 0].set(title="feature means", xlabel="real", ylabel="gen")
axes[0, 0].grid(True, alpha=0.15)

sr, sg = X_real.std(0), X_gen.std(0)
axes[0, 1].scatter(sr, sg, alpha=0.3, s=10, color="#e9c46a")
lim = [min(sr.min(), sg.min()), max(sr.max(), sg.max())]
axes[0, 1].plot(lim, lim, "r--", alpha=0.8)
axes[0, 1].set(title="feature stds", xlabel="real", ylabel="gen")
axes[0, 1].grid(True, alpha=0.15)

if per_traj_fids:
    axes[1, 0].hist(list(per_traj_fids.values()), bins=30, color="#2a9d8f", edgecolor="k", alpha=0.8)
    axes[1, 0].axvline(fid_global, color="red", ls="--", lw=1.5, label=f"global={fid_global:.1f}")
    axes[1, 0].set(title=f"per-traj {fid_label}", xlabel="FID")
    axes[1, 0].legend()
else:
    axes[1, 0].text(0.5, 0.5, "N/A", ha="center", va="center", transform=axes[1, 0].transAxes)

real_norm = X_real / (np.linalg.norm(X_real, axis=1, keepdims=True) + 1e-8)
gen_norm = X_gen / (np.linalg.norm(X_gen, axis=1, keepdims=True) + 1e-8)
cos_per_sample = np.sum(real_norm * gen_norm, axis=1)
axes[1, 1].hist(cos_per_sample, bins=40, color="#2a9d8f", edgecolor="k", alpha=0.8)
axes[1, 1].axvline(cos_per_sample.mean(), color="red", ls="--", lw=1.5,
                    label=f"mean={cos_per_sample.mean():.3f}")
axes[1, 1].set(title="cosine similarity (real vs gen, paired)", xlabel="cosine sim")
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.15)

fig.suptitle(f"{fid_label} = {fid_global:.2f} | mean cos sim = {cos_per_sample.mean():.3f}", fontweight="bold")
fig.tight_layout()

## 3. Warm Restart Evaluation

In [ ]:
from gyaradax import gk_from_gkw_dir

In [ ]:
from neurips_warm_restarts import run_trajectory

N_EVAL_STEPS = 5000
N_RESTARTS   = 5
SCALE_RATIO_FAIL = 5.0
WARM_RESTART_TRAJS = [f.replace(".h5", "") for f in ID_VAL + OOD_VAL] + ["iteration_13"]
GT_SUBSAMPLE = 1
N_GT_TAIL = 3 * 80
RAW_KYSPEC_TO_INTEGRATOR = 16.0

warm_results = {}
_cond_keys = sorted(runner.cfg.model.conditioning)


def _avg_logs(logs):
    """Mean + std of warm-trajectory logs across restarts. Stores
    `<k>_std` (per-step std) and `<k>_per_restart` (R, T, ...) alongside
    the mean for downstream variance diagnostics."""
    keys = [k for k in logs[0] if k != "df_final"]
    out = {}
    for k in keys:
        arrays = [np.asarray(l[k]) for l in logs]
        if k == "time":
            out[k] = arrays[0]
            continue
        stacked = np.stack(arrays, axis=0)  # (R, T, ...)
        out[k] = stacked.mean(axis=0)
        out[f"{k}_std"] = stacked.std(axis=0, ddof=0)
        out[f"{k}_per_restart"] = stacked
    out["n_restarts"] = len(logs)
    return out


for traj in WARM_RESTART_TRAJS:
    print(f"\n{'=' * 88}\nTrajectory {traj}\n{'=' * 88}")
    if traj.startswith("ood_iteration_"):
        _gkw_path = os.path.join(GKW_RAW_DIR, "ood", traj[len("ood_"):])
    else:
        _gkw_path = os.path.join(GKW_RAW_DIR, traj)
    df_gt, geometry, params, state_init, pre = gk_from_gkw_dir(
        _gkw_path, mixed_precision=True, k_index=80,
    )
    _param_map = {"itg": "rlt", "dg": "rln", "s_hat": "shat", "q": "q"}
    cond = torch.tensor(
        [[float(getattr(params, _param_map[k])) for k in _cond_keys]],
        dtype=torch.float32, device=runner.device,
    )
    cond_b = cond.expand(N_RESTARTS, -1).contiguous()

    # Sample N_RESTARTS independent diffusion outputs at this condition.
    with torch.no_grad():
        _decoded_b = runner.sample(cond_b, latent_only=False, steps=N_DENOISING_STEPS)["df"].cpu()
    fi_in_valset = next((idx for idx, fpath in enumerate(runner.valsets[0].files) if traj in fpath), 0)

    gt_max = float(np.max(np.abs(np.asarray(df_gt))))
    gt_std = float(np.std(np.asarray(df_gt).real)) if np.iscomplexobj(np.asarray(df_gt)) else float(np.std(np.asarray(df_gt)))

    restart_logs = []
    df_preds = []
    for ri in range(N_RESTARTS):
        DF_PRED = runner.valsets[0].denormalize(fi_in_valset, df=_decoded_b[ri]).numpy()
        df_warm = from_model_space(DF_PRED)
        df_preds.append(DF_PRED)

        warm_max = float(np.max(np.abs(np.asarray(df_warm))))
        ratio_max = warm_max / max(gt_max, 1e-30)
        if not np.isfinite(ratio_max) or ratio_max > SCALE_RATIO_FAIL:
            print(f"  restart {ri}: ratio_max={ratio_max:.2f} — SKIPPING this restart")
            continue

        log_i = run_trajectory(
            df_warm, geometry, params, pre, state_init,
            N_EVAL_STEPS, f"{traj}/warm{ri}",
            chunk_size=40, backend="jax", mixed_precision=True,
            print_every=500,
        )
        if not np.all(np.isfinite(log_i["eflux"])):
            print(f"  restart {ri}: NaN in eflux — dropping")
            continue
        restart_logs.append(log_i)
        print(f"  restart {ri}: ok  (Q_final={float(log_i['eflux'][-1]):.3e})")

    if not restart_logs:
        print(f"  !! no usable restarts for {traj}; storing empty log_warm.")
        warm_results[traj] = dict(log_warm={}, log_gt_ref={},
                                   ref_flux_samples=None, df_pred=df_preds[0] if df_preds else None,
                                   split=("OOD" if traj.startswith("ood_") else "ID"))
        continue

    log_warm = _avg_logs(restart_logs)
    print(f"  averaged across {len(restart_logs)} restart(s)")

    ref_flux_samples = None
    log_gt_ref = {}
    fluxes_p = os.path.join(_gkw_path, "fluxes.dat")
    if os.path.isfile(fluxes_p):
        ref_flux_samples = np.asarray(
            np.loadtxt(fluxes_p)[::GT_SUBSAMPLE, 1][-N_GT_TAIL:], dtype=np.float64,
        )
    else:
        print(f"  (no fluxes.dat at {fluxes_p})")

    kyspec_p = os.path.join(_gkw_path, "kyspec")
    if os.path.isfile(kyspec_p):
        log_gt_ref["ky_spec"] = np.loadtxt(kyspec_p)[::GT_SUBSAMPLE][-N_GT_TAIL:] * RAW_KYSPEC_TO_INTEGRATOR
    else:
        print(f"  (no kyspec at {kyspec_p})")

    eflux_p = os.path.join(_gkw_path, "eflux_spectra.dat")
    if os.path.isfile(eflux_p):
        log_gt_ref["fluxspec"] = np.loadtxt(eflux_p)[::GT_SUBSAMPLE][-N_GT_TAIL:]
    else:
        print(f"  (no eflux_spectra.dat at {eflux_p})")

    time_p = os.path.join(_gkw_path, "time.dat")
    if os.path.isfile(time_p):
        log_gt_ref["time"] = np.loadtxt(time_p)[::GT_SUBSAMPLE][-N_GT_TAIL:]

    print(f"  GT raw: flux={None if ref_flux_samples is None else ref_flux_samples.shape}  "
          f"ky_spec={getattr(log_gt_ref.get('ky_spec'), 'shape', None)}  "
          f"fluxspec={getattr(log_gt_ref.get('fluxspec'), 'shape', None)}")

    warm_results[traj] = dict(
        log_warm=log_warm, log_gt_ref=log_gt_ref,
        ref_flux_samples=ref_flux_samples, df_pred=df_preds[0],
        n_restarts=len(restart_logs),
        split=("OOD" if traj.startswith("ood_") else "ID"),
    )
    print(f"  {traj}: stored mean-log over {len(restart_logs)} restart(s).")

In [ ]:
# === post-hoc: distributional divergences over stored warm logs ============
# Re-runnable. compute_distribution_divergences detects `<k>_per_restart`
# arrays in log_warm and pools restarts along time for the headline metric,
# while also reporting per-restart spread (mean ± std).
from importlib import reload
import neurips_warm_restarts as _nwr
reload(_nwr)
from neurips_warm_restarts import compute_distribution_divergences

WARM_FRAC = 0.95

PRINT_KEYS = (
    "flux_w1", "flux_mmd", "flux_ks_p", "flux_ad", "flux_arima_l2", "flux_r2_hist",
    "ky_spec_w1_mean", "ky_spec_mmd_mean", "ky_spec_ks_p_mean",
    "ky_spec_ad_mean", "ky_spec_arima_l2_mean", "ky_spec_r2_meanlog",
    "fluxspec_w1_mean", "fluxspec_mmd_mean", "fluxspec_ks_p_mean",
    "fluxspec_ad_mean", "fluxspec_arima_l2_mean", "fluxspec_r2_meanlog",
)

# warm vs GT length sanity print
print(f"{'traj':<22}  {'warm_T':>7}  {'n_r':>3}  {'gt_T':>5}")
for traj, res in warm_results.items():
    lw = res.get("log_warm", {})
    eflux = np.asarray(lw.get("eflux", []))
    if eflux.size == 0:
        continue
    n_r = lw.get("n_restarts", 1)
    gt = res.get("ref_flux_samples")
    print(f"{traj:<22}  {len(eflux):>7}  {n_r:>3}  {len(gt) if gt is not None else 0:>5}")

for traj, res in warm_results.items():
    log_warm = res.get("log_warm")
    if not log_warm or "eflux" not in log_warm:
        res["div_warm"] = {}
        continue
    res["div_warm"] = compute_distribution_divergences(
        log_warm,
        res.get("log_gt_ref", {}),
        ref_flux_samples=res.get("ref_flux_samples"),
        warm_frac=WARM_FRAC,
    )
    print(f"\n  {traj} divergences (warm vs gt saturated):  6 metrics x 3 quantities")
    print(f"  {'metric':32s}  {'pooled':>14s}  {'<per-restart>':>14s}  {'std/<>':>10s}")
    for k in PRINT_KEYS:
        v = res["div_warm"].get(k)
        if v is None or not np.isfinite(v):
            continue
        m = res["div_warm"].get(f"{k}_per_restart_mean", float("nan"))
        s = res["div_warm"].get(f"{k}_per_restart_std",  float("nan"))
        rel = s / abs(m) if (np.isfinite(m) and abs(m) > 1e-30) else float("nan")
        m_str   = f"{m:>14.4g}" if np.isfinite(m) else f"{'--':>14s}"
        rel_str = f"{rel:>10.3f}" if np.isfinite(rel) else f"{'--':>10s}"
        print(f"  {k:32s}  {v:>14.4g}  {m_str}  {rel_str}")


In [ ]:
# === restart-variance diagnostic =========================================
# Bounds the noise floor on warm metrics (driven by N_RESTARTS). If the
# coefficient of variation across restarts is comparable to the spread of
# AD/MMD/KS across trajectories, weak FID-vs-AD correlation is just within-
# trajectory noise and not a missing physics signal — bump N_RESTARTS.
import numpy as np

print("=== Across-restart spread of post-saturation observables ===")
print(f"{'traj':<22}  {'n_r':>3}  {'<Q>':>10}  {'std(Q)/<Q>':>11}  "
        f"{'<W(k_y)> CV':>12}  {'<Q(k_y)> CV':>12}")
for traj, res in warm_results.items():
    log_w = res.get("log_warm", {})
    if "eflux_per_restart" not in log_w:
        print(f"{traj:<22}  -- no per-restart logs (re-run solver) --")
        continue
    e_pr = np.asarray(log_w["eflux_per_restart"])           # (R, T)
    e_late = e_pr[:, e_pr.shape[1] // 2:]                    # post-saturation half
    e_mean = e_late.mean()
    e_cv = e_late.std() / max(abs(e_mean), 1e-30)
    cv_ky  = (np.asarray(log_w["ky_spec_per_restart"])[:, e_pr.shape[1]//2:].std(axis=0).mean()
              / max(abs(np.asarray(log_w["ky_spec"]).mean()), 1e-30)) \
              if "ky_spec_per_restart"  in log_w else float("nan")
    cv_qky = (np.asarray(log_w["fluxspec_per_restart"])[:, e_pr.shape[1]//2:].std(axis=0).mean()
              / max(abs(np.asarray(log_w["fluxspec"]).mean()), 1e-30)) \
              if "fluxspec_per_restart" in log_w else float("nan")
    print(f"{traj:<22}  {log_w['n_restarts']:>3}  {e_mean:>10.3e}  {e_cv:>11.3f}  "
          f"{cv_ky:>12.3f}  {cv_qky:>12.3f}")
print("\nrule of thumb: if std/<Q> on flux is >= 0.1 the AD/MMD/KS spread across "
       "trajs is dominated by within-traj noise — bump N_RESTARTS.")


In [ ]:
n = len(warm_results)
fig, axes = plt.subplots(n, 4, figsize=(22, 4 * n), squeeze=False)
snap_colors = ["#2a9d8f", "#e76f51", "#264653"]

for row, (it, res) in enumerate(warm_results.items()):
    lw = res["log_warm"]
    t = lw["time"]
    ref_flux = res["ref_flux_samples"]
    log_gt_ref = res["log_gt_ref"]

    n_t = len(t)
    snap_idx = [min(5, n_t - 1), n_t // 2, n_t - 1]

    # col 0: ky-spectrum: 3 warm snapshots + GT-saturated mean
    for j, si in enumerate(snap_idx):
        ky_w = np.log10(np.maximum(lw["ky_spec"][si], 1e-30))
        axes[row, 0].plot(ky_w, "-", color=snap_colors[j], lw=1.2, alpha=0.9,
                           label=f"warm t={float(t[si]):.1f}")
    if "ky_spec" in log_gt_ref:
        ky_gt_mean = np.log10(np.maximum(np.asarray(log_gt_ref["ky_spec"]).mean(0), 1e-30))
        axes[row, 0].plot(ky_gt_mean, "k--", lw=1.4, alpha=0.85, label="GT mean")
    axes[row, 0].set(title=f"iter {it}: $W(k_y)$", xlabel="$k_y$ mode",
                       ylabel="log$_{10}$ W")
    axes[row, 0].legend(fontsize=7); axes[row, 0].grid(True, alpha=0.15)

    # col 1: fluxspec in linear (signed) — kyspec is log
    if "fluxspec" in lw:
        for j, si in enumerate(snap_idx):
            fs_w = np.asarray(lw["fluxspec"][si])
            axes[row, 1].plot(fs_w, "-", color=snap_colors[j], lw=1.2, alpha=0.9,
                               label=f"warm t={float(t[si]):.1f}")
        if "fluxspec" in log_gt_ref:
            fs_gt_mean = np.asarray(log_gt_ref["fluxspec"]).mean(0)
            axes[row, 1].plot(fs_gt_mean, "k--", lw=1.4, alpha=0.85, label="GT mean")
        axes[row, 1].axhline(0, color="gray", lw=0.5, alpha=0.4)
        axes[row, 1].set(title=f"iter {it}: $Q(k_y)$", xlabel="$k_y$ mode",
                           ylabel="$Q$")
        axes[row, 1].legend(fontsize=7); axes[row, 1].grid(True, alpha=0.15)
    else:
        axes[row, 1].text(0.5, 0.5, "no fluxspec", ha="center", va="center",
                            transform=axes[row, 1].transAxes); axes[row, 1].set_axis_off()

    # col 2: flux trace + GT reference band
    axes[row, 2].plot(t, lw["eflux"], lw=1, color="#2a9d8f", label="warm")
    if ref_flux is not None and len(ref_flux) > 1:
        m, s = float(np.mean(ref_flux)), float(np.std(ref_flux))
        axes[row, 2].axhspan(m - s, m + s, color="k", alpha=0.08, label="GT ±1σ")
        axes[row, 2].axhline(m, color="k", ls=":", lw=0.8)
    axes[row, 2].set_title(f"iter {it}: flux"); axes[row, 2].legend(fontsize=7)
    axes[row, 2].grid(True, alpha=0.15)

    # col 3: Pearson(log ky_warm, log ky_GT_mean) over time
    if "ky_spec" in log_gt_ref:
        ky_gt_mean = np.log10(np.maximum(np.asarray(log_gt_ref["ky_spec"]).mean(0), 1e-30))
        n_compare = len(lw["ky_spec"])
        r_ts = []
        for i in range(n_compare):
            ky_w = np.log10(np.maximum(lw["ky_spec"][i], 1e-30))
            r_ts.append(pearsonr(ky_w, ky_gt_mean)[0] if len(ky_gt_mean) > 1 else 0.0)
        axes[row, 3].plot(t[:n_compare], r_ts, lw=1, label="Pearson")
        axes[row, 3].axhline(0.95, color="gray", ls="--", lw=0.8)
        axes[row, 3].set_title(f"iter {it}: Pearson($k_y$ warm, GT mean)")
        axes[row, 3].set_ylim(-0.1, 1.05)
        axes[row, 3].legend(fontsize=7); axes[row, 3].grid(True, alpha=0.15)
    else:
        axes[row, 3].text(0.5, 0.5, "no kyspec ref", ha="center", va="center",
                            transform=axes[row, 3].transAxes); axes[row, 3].set_axis_off()

for ax in axes[-1]:
    ax.set_xlabel(r"time $[v_{th}/R]$")
fig.tight_layout()

In [ ]:
import pickle as _pickle
from neugk.utils import separate_zf as _sep_zf

K_GEN = 256          # generated samples per traj (chunked by GEN_BATCH_SIZE under the hood)
N_GT_SAMPLES = None  # None = use ALL preprocessed bins for that traj

LATENT_LEVELS = {
    "skip_deep":  {"source": "skip",       "decoder_level": -1, "pool": "amax"},
    "bottleneck": {"source": "bottleneck", "pool": "amax"},
    "flux_head":  {"source": "flux_head",  "flux_head_level": 1},
    "phi":        {"source": "phi",        "pool": "amax"},
}

_gs_mean_t = torch.as_tensor(np.asarray(gs_stats["df"]["full"]["mean"]), dtype=torch.float32)
_gs_std_t  = torch.as_tensor(np.asarray(gs_stats["df"]["full"]["std"]),  dtype=torch.float32)
_gs_nontime_local = [k for k in gs_cond_keys if k != "timestep"]
_diff_cond_keys   = sorted(runner.cfg.model.conditioning)
_meta_map_local   = {"itg": "ion_temp_grad", "dg": "density_grad"}


def _gs_norm(df_model):
    m = _gs_mean_t.view(_gs_mean_t.shape + (1,) * (df_model.ndim - _gs_mean_t.ndim))
    s = _gs_std_t.view(_gs_std_t.shape  + (1,) * (df_model.ndim - _gs_std_t.ndim))
    return (df_model - m) / s


def _build_gt_latents_id(traj, levels, n_samples=N_GT_SAMPLES):
    """`n_samples` (or all if None) preprocessed bins -> GyroSwin features per
    level, BATCHED in chunks of GEN_BATCH_SIZE for speed."""
    gt_dir = os.path.join(DATA_PATH, f"{traj}_ifft_realpotens")
    meta_p = os.path.join(gt_dir, "metadata.pkl")
    if not os.path.isfile(meta_p):
        return None
    with open(meta_p, "rb") as f:
        meta = _pickle.load(f)
    n_ts = len(meta["timesteps"]); res = meta["resolution"]
    if n_samples is None or n_samples > n_ts:
        n_samples = n_ts
    cond_vals = torch.tensor(
        [[float(np.squeeze(meta[_meta_map_local.get(k, k)])) for k in _gs_nontime_local]],
        dtype=torch.float32,
    )
    sep_zf = bool(runner.cfg.dataset.separate_zf)

    # First load all bin tensors into a list, then batch them through the
    # extractor so the GyroSwin forward sees mini-batches instead of N=1.
    all_dfs = []
    for t in range(max(0, n_ts - n_samples), n_ts):
        bin_path = os.path.join(gt_dir, "data", f"timestep_{t:05d}.bin")
        if not os.path.isfile(bin_path):
            continue
        arr = np.fromfile(bin_path, dtype=np.float32).reshape(2, *res)
        if sep_zf:
            arr = _sep_zf(arr, dim=0)
        all_dfs.append(_gs_norm(torch.as_tensor(arr, dtype=torch.float32)))
    if not all_dfs:
        return None

    out = {name: [] for name in levels}
    for i in range(0, len(all_dfs), GEN_BATCH_SIZE):
        batch = torch.stack(all_dfs[i:i + GEN_BATCH_SIZE]).to(runner.device)
        cond_batch = cond_vals.expand(batch.shape[0], -1).contiguous()
        for name, kwargs in levels.items():
            kw = dict(kwargs)
            if kw["source"] in ("flux_head", "decoder", "skip", "phi"):
                kw["cond_keys"] = _gs_nontime_local
            try:
                feats = extract_gyroswin_latents(
                    gyroswin_model, batch, device=runner.device,
                    condition=cond_batch, **kw,
                )
            except Exception:
                feats = np.zeros((batch.shape[0], 1), dtype=np.float32)
            out[name].append(feats)
    return {name: np.concatenate(v, axis=0) for name, v in out.items() if v}


def _generate_traj_samples_id(traj, k_gen=K_GEN):
    """Draw `k_gen` diffusion samples for `traj`'s condition vector and
    return them in gs-normalised space, ready for the GyroSwin extractor.
    Already chunked by GEN_BATCH_SIZE inside the loop."""
    fi = next((idx for idx, fpath in enumerate(runner.valsets[0].files)
                if traj in fpath), None)
    if fi is None:
        return None, None
    meta_diff = runner.valsets[0].metadata[fi]
    diff_cond = torch.tensor(
        [float(np.squeeze(meta_diff[_meta_map_local.get(k, k)])) for k in _diff_cond_keys],
        dtype=torch.float32,
    )
    gs_cond = torch.tensor(
        [float(np.squeeze(meta_diff[_meta_map_local.get(k, k)])) for k in _gs_nontime_local],
        dtype=torch.float32,
    )
    runner.model.eval()
    gen_dfs = []
    n_done = 0
    while n_done < k_gen:
        b = min(GEN_BATCH_SIZE, k_gen - n_done)
        c = diff_cond.unsqueeze(0).expand(b, -1).contiguous().to(runner.device)
        with torch.no_grad():
            out = runner.sample(c, steps=N_DENOISING_STEPS, latent_only=False)
        for j in range(out["df"].shape[0]):
            df_diff_norm = out["df"][j].cpu()
            df_raw = runner.valsets[0].denormalize(fi, df=df_diff_norm)
            gen_dfs.append(_gs_norm(df_raw).to(df_diff_norm.dtype))
        n_done += b
    return gen_dfs, gs_cond


def _gen_features(gen_dfs, gs_cond, levels):
    feats_per_level = {name: [] for name in levels}
    cond_batch_template = gs_cond.unsqueeze(0)
    for i in range(0, len(gen_dfs), GEN_BATCH_SIZE):
        batch_df = torch.stack(gen_dfs[i:i + GEN_BATCH_SIZE])
        b = batch_df.shape[0]
        cond_batch = cond_batch_template.expand(b, -1).contiguous()
        for name, kwargs in levels.items():
            kw = dict(kwargs)
            if kw["source"] in ("flux_head", "decoder", "skip", "phi"):
                kw["cond_keys"] = _gs_nontime_local
            try:
                f = extract_gyroswin_latents(
                    gyroswin_model, batch_df, device=runner.device,
                    condition=cond_batch, **kw,
                )
            except Exception:
                f = np.zeros((b, 1), dtype=np.float32)
            feats_per_level[name].append(f)
    return {name: np.concatenate(v, axis=0) for name, v in feats_per_level.items() if v}


def _mean_pairwise_cosine(A, B):
    """Mean over all (a_i, b_j) pairs of <a_i, b_j> / (||a_i|| ||b_j||).
    A: (Na, D), B: (Nb, D)."""
    An = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
    Bn = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-12)
    return float((An @ Bn.T).mean())


trajs = list(warm_results.keys())
levels = list(LATENT_LEVELS.keys())
per_iter_fids = {}
per_iter_cos  = {}

for traj in trajs:
    gt_per_level = _build_gt_latents_id(traj, LATENT_LEVELS)
    if gt_per_level is None:
        print(f"  {traj}: no preprocessed metadata; skipping.")
        continue
    gen_dfs, gs_cond = _generate_traj_samples_id(traj)
    if gen_dfs is None:
        print(f"  {traj}: no valset match; skipping.")
        continue
    gen_per_level = _gen_features(gen_dfs, gs_cond, LATENT_LEVELS)

    iter_fids = {}
    iter_cos  = {}
    for lv in levels:
        if lv not in gen_per_level or lv not in gt_per_level:
            iter_fids[lv] = float("nan"); iter_cos[lv] = float("nan"); continue
        A = gen_per_level[lv]
        B = gt_per_level[lv]
        if A.ndim != 2 or B.ndim != 2 or A.shape[1] != B.shape[1]:
            iter_fids[lv] = float("nan"); iter_cos[lv] = float("nan"); continue

        # FID with PCA reduction; report explained-variance-ratio for honesty
        n_comp = max(2, min(A.shape[1], min(A.shape[0], B.shape[0]) - 1))
        try:
            pooled = np.concatenate([A, B], axis=0)
            pca = PCA(n_components=min(n_comp, pooled.shape[1])).fit(pooled)
            Ap = pca.transform(A)
            Bp = pca.transform(B)
            ev = float(pca.explained_variance_ratio_.sum())
        except Exception:
            Ap, Bp = A, B
            ev = float("nan")
        mu_g, sig_g = compute_statistics(Ap)
        mu_r, sig_r = compute_statistics(Bp)
        iter_fids[lv] = float(compute_fid(mu_g, sig_g, mu_r, sig_r))

        # Cosine similarity on raw (unprojected) features.
        iter_cos[lv] = _mean_pairwise_cosine(A, B)

        print(f"    {traj}/{lv:11s}  N_gen={A.shape[0]} N_gt={B.shape[0]} "
              f"D={A.shape[1]} pca={Ap.shape[1]} ev={ev:.1%}  "
              f"FID={iter_fids[lv]:.3f}  cos={iter_cos[lv]:+.3f}")

    per_iter_fids[traj] = iter_fids
    per_iter_cos[traj]  = iter_cos
    print(f"  {traj}: " + "  ".join(
        f"FID({lv})={iter_fids[lv]:.3f}/cos={iter_cos[lv]:+.3f}" for lv in levels))


In [ ]:
import numpy as np

if "_generate_traj_samples_id" not in dir() or "_build_gt_latents_id" not in dir():
    raise RuntimeError("run `latent_dashboard` first to define the helpers")

SPLIT_TRAJ  = next((t for t in warm_results if not t.startswith("ood_")), None)
SPLIT_K_GEN = max(2 * K_GEN, 128)

if SPLIT_TRAJ is None:
    print("no ID traj for split-half FID")
else:
    print(f"=== Split-half FID noise on {SPLIT_TRAJ} (K_GEN={SPLIT_K_GEN}, halved) ===")
    gt_per_level = _build_gt_latents_id(SPLIT_TRAJ, LATENT_LEVELS)
    gen_dfs, gs_cond = _generate_traj_samples_id(SPLIT_TRAJ, k_gen=SPLIT_K_GEN)
    half = len(gen_dfs) // 2
    feats_full = _gen_features(gen_dfs,           gs_cond, LATENT_LEVELS)
    feats_a    = _gen_features(gen_dfs[:half],    gs_cond, LATENT_LEVELS)
    feats_b    = _gen_features(gen_dfs[half:],    gs_cond, LATENT_LEVELS)
    print(f"  {'level':<11}  {'FID_full':>9}  {'FID_A':>9}  {'FID_B':>9}  "
            f"{'|A-B|':>9}  {'|A-B|/full':>11}")
    def _fid(A, B):
        n = max(2, min(A.shape[1], min(A.shape[0], B.shape[0]) - 1))
        try:
            pca = PCA(n_components=min(n, A.shape[1])).fit(np.concatenate([A, B], 0))
            A, B = pca.transform(A), pca.transform(B)
        except Exception:
            pass
        ma, sa = compute_statistics(A); mb, sb = compute_statistics(B)
        return float(compute_fid(ma, sa, mb, sb))
    for lv in LATENT_LEVELS:
        if lv not in feats_full or lv not in gt_per_level:
            continue
        f_full = _fid(feats_full[lv], gt_per_level[lv])
        f_a    = _fid(feats_a[lv],    gt_per_level[lv])
        f_b    = _fid(feats_b[lv],    gt_per_level[lv])
        gap    = abs(f_a - f_b)
        rel    = gap / max(f_full, 1e-30)
        print(f"  {lv:<11}  {f_full:>9.3g}  {f_a:>9.3g}  {f_b:>9.3g}  "
              f"{gap:>9.3g}  {rel:>11.3f}")
    print("rule of thumb: if |A-B|/full is comparable to traj-to-traj spread of "
           "FID, the per-traj FID is K_GEN-noise-limited — bump K_GEN.")

In [ ]:
# === DUMP simulation state for the analysis notebook =====================
# Persists everything section 4+ analysis needs. Re-runnable. Loaded by
# `neurips_diff_eval_analysis.ipynb`.
import os
import pickle as _pickle

DUMP_PATH = os.path.join(RESULTS_DIR, f"sim_state_{METHOD_NAME}.pkl")
os.makedirs(RESULTS_DIR, exist_ok=True)

dump_payload = {
    # === per-trajectory simulation outputs (heavy) ========================
    "warm_results":     warm_results,         # log_warm + log_gt_ref + ref_flux_samples + df_pred + n_restarts + split per traj
    "per_iter_fids":    per_iter_fids,        # per-(traj, U-Net level) single-shot generative FID
    "per_iter_cos":     globals().get("per_iter_cos", {}),  # mean pairwise cosine of raw features
    # === auxiliary section-2 results (light) ==============================
    "gs_fid_per_source": globals().get("gs_fid_per_source", {}),  # global FID per source
    "paper_probe":       globals().get("PAPER_PROBE", {}),         # probe scatter data
    "latent_levels":     list(LATENT_LEVELS.keys()),
    # === constants needed by analysis cells ===============================
    "constants": {
        "method_name":              METHOD_NAME,
        "results_dir":              RESULTS_DIR,
        "data_path":                DATA_PATH,
        "gkw_raw_dir":              GKW_RAW_DIR,
        "ae_checkpoint":            AE_CHECKPOINT,
        "pretrained_dir":           PRETRAINED_DIR,
        "gyroswin_checkpoint":      GYROSWIN_CHECKPOINT,
        "n_eval_steps":             N_EVAL_STEPS,
        "n_denoising_steps":        N_DENOISING_STEPS,
        "k_gen":                    K_GEN,
        "n_restarts":               N_RESTARTS,
        "n_gt_tail":                N_GT_TAIL,
        "gt_subsample":             GT_SUBSAMPLE,
        "raw_kyspec_to_integrator": RAW_KYSPEC_TO_INTEGRATOR,
        "fid_n_components":         FID_N_COMPONENTS,
    },
}

with open(DUMP_PATH, "wb") as f:
    _pickle.dump(dump_payload, f)

# Sanity report on what we dumped — useful when iterating.
import numpy as _np
print(f"=== dumped {DUMP_PATH} ({os.path.getsize(DUMP_PATH) / 1e6:.1f} MB) ===")
print(f"  trajectories     : {sorted(warm_results.keys())}")
print(f"  per_iter_fids    : {sorted(per_iter_fids.keys())}")
print(f"  latent levels    : {dump_payload['latent_levels']}")
print(f"  constants        : {sorted(dump_payload['constants'].keys())}")
for tj in list(warm_results.keys())[:1]:
    lw = warm_results[tj].get("log_warm", {})
    keys = sorted(lw.keys()) if isinstance(lw, dict) else []
    print(f"  log_warm[{tj}] keys: {keys}")
    log_gt = warm_results[tj].get("log_gt_ref", {})
    print(f"  log_gt_ref[{tj}] keys: {sorted(log_gt.keys())}")
